# 340 · Resonator spectroscopy in parallel

**Level:** advanced · **Topics:** Gen2, SHFQA, spectroscopy, readout, sweeps, calibration · **Type:** fix the bugs

A user has four qubits on two feedlines: `q0` and `q1` share readout channel 0, `q2` and `q3` share readout channel 1.

To save time, the user wrote a resonator spectroscopy that sweeps the frequency of **all four resonators at the same time**,
and asks whether it works. It does not.

Do not change the wiring. Get the spectroscopy of all four resonators, with as much in parallel as the instrument allows.

Everything runs in emulation, no instrument needed.

In [ ]:
from laboneq.simple import *

## Device setup

One SHFQA+ with four readout channels. Two are used, each with two qubits.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
qubits = ["q0", "q1", "q2", "q3"]

setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(SHFQA(uid="shfqa", address="DEV12001", device_options="SHFQA4"))
setup.add_connections(
    "shfqa",
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
    create_connection(to_signal="q1/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q1/acquire", ports="QACHANNELS/0/INPUT"),
    create_connection(to_signal="q2/measure", ports="QACHANNELS/1/OUTPUT"),
    create_connection(to_signal="q2/acquire", ports="QACHANNELS/1/INPUT"),
    create_connection(to_signal="q3/measure", ports="QACHANNELS/1/OUTPUT"),
    create_connection(to_signal="q3/acquire", ports="QACHANNELS/1/INPUT"),
)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## The spectroscopy of the user

`spectroscopy(qubits)` builds one experiment that sweeps the readout frequency of all the given qubits at the same time.
Every qubit has its own sweep, 20 MHz wide. The oscillators are set in the calibration of the experiment.

The function itself is fine, there is nothing to change in it.

📖 [Frequency sweep examples](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/03_frequency_sweep_examples.html) ·
[Experiment calibration](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/concepts/00_experiment_calibration.html) ·
[Resonator spectroscopy](https://docs.zhinst.com/labone_q_user_manual/applications_library/how-to-guides/sources/01_superconducting_qubits/01_workflows/01_resonator_spectroscopy.html)

In [ ]:
LO_FREQUENCY = 7.0e9
SWEEP_START = {"q0": -150e6, "q1": 60e6, "q2": -90e6, "q3": 120e6}

readout_pulse = pulse_library.const(length=2e-6, amplitude=0.1)


def spectroscopy(qubits):
    signals = []
    sweeps = []
    for qubit in qubits:
        signals.append(f"{qubit}_measure")
        signals.append(f"{qubit}_acquire")
        start = SWEEP_START[qubit]
        sweeps.append(
            LinearSweepParameter(uid=f"frequency_{qubit}", start=start, stop=start + 20e6, count=21)
        )

    @dsl.experiment(signals=signals)
    def experiment():
        with dsl.acquire_loop_rt(count=16, acquisition_type=AcquisitionType.SPECTROSCOPY):
            with dsl.sweep(parameter=sweeps):
                for qubit in qubits:
                    with dsl.section(name=f"measure_{qubit}"):
                        dsl.measure(
                            measure_signal=f"{qubit}_measure",
                            acquire_signal=f"{qubit}_acquire",
                            measure_pulse=readout_pulse,
                            integration_length=2e-6,
                            handle=f"{qubit}/result",
                            reset_delay=1e-6,
                        )

        experiment_calibration = dsl.experiment_calibration()
        for qubit, sweep in zip(qubits, sweeps):
            oscillator = Oscillator(frequency=sweep, modulation_type=ModulationType.HARDWARE)
            experiment_calibration[f"{qubit}_measure"] = SignalCalibration(
                oscillator=oscillator,
                local_oscillator=Oscillator(frequency=LO_FREQUENCY),
                range=-10,
            )
            experiment_calibration[f"{qubit}_acquire"] = SignalCalibration(
                oscillator=oscillator,
                local_oscillator=Oscillator(frequency=LO_FREQUENCY),
                range=0,
            )

    exp = experiment()
    for qubit in qubits:
        exp.map_signal(f"{qubit}_measure", f"{qubit}/measure")
        exp.map_signal(f"{qubit}_acquire", f"{qubit}/acquire")
    return exp

## Run it

All four qubits in one experiment. This is the cell to fix.

In [ ]:
data = {}

compiled = session.compile(spectroscopy(qubits))
results = session.run(compiled)
for qubit in qubits:
    data[qubit] = results[f"{qubit}/result"].data

In [ ]:
# check
for qubit in qubits:
    print(qubit, data[qubit].shape)

**Expected result:**

```
q0 (21,)
q1 (21,)
q2 (21,)
q3 (21,)
```

## Questions

1. What does the error say, and what is the limit behind it?
2. How many resonators can be measured in parallel with this instrument?
3. The error message mentions the LRT option. Would it solve the problem?
4. Could the two sweeps of one channel run one after the other inside a single experiment?

<details>
<summary>Hint 1</summary>

Try `spectroscopy(["q0", "q2"])` and `spectroscopy(["q0", "q1"])`. Which one compiles?
</details>

<details>
<summary>Hint 2</summary>

`spectroscopy` accepts any list of qubits. Nothing forces you to pass all four at once.
</details>

The solution is in [solution.ipynb](solution.ipynb).